In [ ]:
# Cell 1: Imports
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

from ultralytics import YOLO
import wandb
import torch

# Cell 2: Check GPU
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")

# Cell 3: Initialize wandb (optional)
# wandb.init(project="license-plate-detection", name="yolo-training")

# Cell 4: Load model
model = YOLO('yolov8n.pt')  # Start from pretrained

# Cell 5: Train model
results = model.train(
    data='data/processed/dataset/data.yaml',  # Your dataset path
    epochs=100,
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=4,
    project='runs/train',
    name='license_plate_detector',
    exist_ok=True,
    patience=10,
    save=True,
    save_period=10,
    verbose=True
)

# Cell 6: Evaluate
metrics = model.val()
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"mAP50: {metrics.box.map50:.4f}")

# Cell 7: Export model
model.export(format='onnx')  # Export to ONNX